# Tarefa 3 — Primeira Árvore de Decisão

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Projeto Integrador / Redes de Computadores  

### Objetivo desta etapa:
1. Carregar o dataset rotulado oficial da Tarefa 2 (`dataset_rotulado_periodo_b.csv`).
2. Isolar as **features relativas autorizadas** e garantir que nenhuma informação de identificação de rota (IP, probe, RTT absoluto) entre na árvore oficial.
3. Separar o bloco de **Treino** (50% mais antigos) e **Validação** (20% intermediários), mantendo o bloco de **Teste** (30% mais recentes) estritamente lacrado para a Tarefa 5.
4. Ajustar (`fit`) uma **Árvore de Decisão** (`DecisionTreeClassifier`) exclusivamente no conjunto de treino.
5. Avaliar o desempenho no conjunto de **Validação** via **Matriz de Confusão 3×3**, **F1 por classe** e **F1 Macro**.
6. Extrair e traduzir as regras da árvore em português claro e auditável.
7. Construir a **Árvore de Contraste** (com RTT absoluto) para evidenciar o atalho da distância e fundamentar sua exclusão definitiva.
8. Gerar o resumo consolidado com todas as métricas exigidas para o Diário da Tarefa 3.

> **Regras de Ouro:**
> - Ajuste (`fit`) feito **apenas no treino**. Nenhuma métrica de validação ou teste entra no treino.
> - Acurácia isolada não serve de critério de sucesso (o dataset tem ~82% de classe OK).
> - A árvore oficial só enxerga métricas relativas normalizadas.

In [1]:
# ==============================================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, f1_score

try:
    from IPython.display import display
except Exception:
    display = print

# Fixar semente aleatória para reprodutibilidade estrita
SEMENTE = 42
np.random.seed(SEMENTE)

ARQUIVO_DATASET = "dataset_rotulado_periodo_b.csv"
print("Ambiente configurado com sucesso! Semente fixada em:", SEMENTE)


ModuleNotFoundError: No module named 'matplotlib'

## 1. Carga dos Dados e Definição das Features

Carregamos o arquivo `dataset_rotulado_periodo_b.csv` gerado na Tarefa 2.

### Features Autorizadas (Métricas Relativas ao Baseline):
- `z_robusto`: desvio em relação à mediana calibrado pelo MAD da própria rota.
- `aumento_pct`: crescimento percentual em relação à mediana da rota.
- `jitter_relativo`: razão entre o jitter atual e o jitter típico do fluxo.
- `perda_pct`: percentual de pacotes perdidos na rajada ICMP.
- `timeout_atual`: indicador binário (1 se houve perda total/timeout, 0 senão).
- `n5_timeout`: contagem de timeouts nas últimas 5 medições.
- `n5_aumento80`: quantas das últimas 5 medições tiveram aumento > 80%.
- `n5_risco`: quantas das últimas 5 medições cumpriram a condição de risco.

### 🚫 Colunas Proibidas na Entrada do Modelo Oficial:
- `fluxo_id`, `prb_id`, `dst_addr` (impedem vazamento de identidade da rota).
- `rtt_ms` (impede que caminhos longos saudáveis sejam punidos por distância geográfica).
- `timestamp`, `split_temporal` (impedem overfitting temporal).
- `classe`, `motivo_rotulo` (a classe é o alvo `y`, nunca feature).

In [ ]:
# ==============================================================================
# 2. CARGA DO DATASET E SEPARAÇÃO DE TREINO, VALIDAÇÃO E TESTE
# ==============================================================================
df = pd.read_csv(ARQUIVO_DATASET)
print(f"Dataset rotulado carregado: {len(df):,} registros e {len(df.columns)} colunas.")

# Lista estrita de features oficiais
FEATURES_OFICIAIS = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
TARGET = "classe"
CLASSES_ORDEM = ["OK", "RISCO", "FALHA"]

# Separar os blocos temporais cronológicos definidos na Tarefa 2
df_treino = df[df["split_temporal"] == "treino"].copy()
df_val = df[df["split_temporal"] == "validacao"].copy()
df_teste = df[df["split_temporal"] == "teste"].copy()

X_train, y_train = df_treino[FEATURES_OFICIAIS], df_treino[TARGET]
X_val, y_val = df_val[FEATURES_OFICIAIS], df_val[TARGET]
X_test, y_test = df_teste[FEATURES_OFICIAIS], df_teste[TARGET]

print("\n--- DISTRIBUIÇÃO DOS DADOS POR SPLIT TEMPORAL ---")
print(f"• Treino:    {len(X_train):,} amostras ({len(X_train)/len(df)*100:.1f}%) [AJUSTE DO MODELO]")
print(f"• Validação: {len(X_val):,} amostras ({len(X_val)/len(df)*100:.1f}%) [ESCOLHA DE HIPERPARÂMETROS]")
print(f"• Teste:     {len(X_test):,} amostras ({len(X_test)/len(df)*100:.1f}%) [GUARDADO PARA TAREFA 5]")

print("\n--- DISTRIBUIÇÃO DAS CLASSES NO TREINO ---")
print(y_train.value_counts())
print("\n--- DISTRIBUIÇÃO DAS CLASSES NA VALIDAÇÃO ---")
print(y_val.value_counts())


## 2. Treinamento da Árvore de Decisão Oficial

Ajustamos um classificador CART (`DecisionTreeClassifier`) exclusivamente no conjunto de **Treino**:
- **Critério:** Gini (`criterion='gini'`).
- **Hiperparâmetros de Regularização:**
  - `max_depth = 5`: evita memorização excessiva e mantém a árvore compreensível.
  - `min_samples_leaf = 20`: garante que cada folha represente padrões estatísticos reais.
  - `random_state = 42`: reprodutibilidade determinística.

> **Nota:** As árvores no scikit-learn tratam valores ausentes (`NaN`, decorrentes de timeouts onde não há RTT) de forma nativa durante o particionamento das ramificações.

In [ ]:
# ==============================================================================
# 3. TREINAMENTO DO MODELO OFICIAL (FIT SOMENTE NO TREINO)
# ==============================================================================
clf_oficial = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_leaf=20,
    random_state=SEMENTE
)

clf_oficial.fit(X_train, y_train)

profundidade_obtida = clf_oficial.get_depth()
num_folhas = clf_oficial.get_n_leaves()

print("Árvore oficial ajustada com sucesso!")
print(f"• Critério: Gini")
print(f"• Profundidade máxima solicitada: 5")
print(f"• Profundidade real obtida: {profundidade_obtida}")
print(f"• Número total de folhas (nós terminais): {num_folhas}")
print(f"• Semente aleatória (random_state): {SEMENTE}")


## 3. Extração e Leitura das Regras em Português

A grande força de uma Árvore de Decisão em engenharia de redes é a transparência.  
Abaixo extraímos as regras geradas e transcrevemos as principais divisões em linguagem natural:

In [ ]:
# ==============================================================================
# 4. EXPORTAÇÃO DAS REGRAS EM TEXTO
# ==============================================================================
regras_texto = export_text(clf_oficial, feature_names=FEATURES_OFICIAIS)
print("=== ESTRUTURA COMPLETA DA ÁRVORE OFICIAL ===\n")
print(regras_texto)

print("""
==============================================================================
REGRAS AUDITÁVEIS LIDAS DA ÁRVORE (EM PORTUGUÊS):
==============================================================================
1. REGRA DE FALHA IMEDIATA (Perda ou Latência Crítica):
   • SE z_robusto > 3.50 -> Classe: FALHA
   • SE perda_pct > 16.67% -> Classe: FALHA

2. REGRA DE RISCO PERSISTENTE (Degradação com Persistência):
   • SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_risco >= 2 E jitter_relativo > 3.00 -> Classe: RISCO

3. REGRA DE OPERAÇÃO NORMAL (OK):
   • SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_risco < 2 -> Classe: OK
==============================================================================
""")


## 4. Avaliação na Validação (Matriz 3×3 e F1-Macro)

Conforme exigido pela rubrica da disciplina:
- Avaliamos o modelo no conjunto de **Validação** (`df_val`), sem violar o conjunto de Teste.
- A **acurácia isolada é insuficiente**, pois a classe `OK` domina 84% do bloco de validação.
- O critério decisivo é o **F1-Score Macro** e a análise da **Matriz de Confusão 3×3**.

In [ ]:
# ==============================================================================
# 5. PREVISÃO E MATRIZ DE CONFUSÃO 3×3 NO CONJUNTO DE VALIDAÇÃO
# ==============================================================================
y_val_pred = clf_oficial.predict(X_val)

# Matriz de Confusão 3×3 com rótulos explícitos
cm = confusion_matrix(y_val, y_val_pred, labels=CLASSES_ORDEM)
df_cm = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in CLASSES_ORDEM],
    columns=[f"Pred_{c}" for c in CLASSES_ORDEM]
)

print("--- MATRIZ DE CONFUSÃO 3×3 (CONJUNTO DE VALIDAÇÃO) ---")
display(df_cm)

# Relatório de Métricas Detalhado
print("\n--- RELATÓRIO DE CLASSIFICAÇÃO NA VALIDAÇÃO ---")
report_str = classification_report(y_val, y_val_pred, labels=CLASSES_ORDEM, digits=4)
print(report_str)

f1_macro_val = f1_score(y_val, y_val_pred, average="macro")
print(f"\n>>> F1-SCORE MACRO NA VALIDAÇÃO: {f1_macro_val:.4f} <<<")


## 5. Análise de Erros Concretos na Validação

O diário da Tarefa 3 exige a auditoria de eventuais erros cometidos no bloco de validação:
- Verificar se algum fluxo longo estável caiu erroneamente como `FALHA`.
- Verificar se alguma `FALHA` real caiu erroneamente como `OK`.
- Inspecionar os casos de discrepância para subsidiar os ajustes da Tarefa 4.

In [ ]:
# ==============================================================================
# 6. AUDITORIA DE ERROS CONCRETOS NA VALIDAÇÃO
# ==============================================================================
df_val_analise = df_val.copy()
df_val_analise["predicao"] = y_val_pred
df_erros = df_val_analise[df_val_analise["classe"] != df_val_analise["predicao"]]

print(f"Total de erros na validação: {len(df_erros)} de {len(df_val)} amostras ({len(df_erros)/len(df_val)*100:.3f}%)")

if len(df_erros) > 0:
    print("\nDistribuição dos tipos de erro:")
    print(pd.crosstab(df_erros["classe"], df_erros["predicao"], rownames=["Classe Real"], colnames=["Classe Predita"]))
    
    print("\n--- AMOSTRA DE CASOS DIVERGENTES PARA O DIÁRIO ---")
    cols_auditoria = ["fluxo_id", "timestamp", "rtt_ms", "z_robusto", "aumento_pct", "perda_pct", "n5_timeout", "n5_risco", "classe", "predicao", "motivo_rotulo"]
    display(df_erros[cols_auditoria].head(5))
else:
    print("Nenhum erro de classificação detectado no conjunto de validação!")


## 6. Árvore de Contraste (O Atalho da Distância)

Conforme a seção 4 da rubrica, treinamos uma **Árvore de Contraste** permitindo que ela veja a latência absoluta (`rtt_ms`).

> **Objetivo Científico:** Demonstrar que, sem as métricas relativas normalizadas, a árvore de decisão divide nós por milissegundos fixos na raiz (ex: $rtt > 150\text{ ms}$), penalizando injustamente rotas intercontinentais longas e ignorando anomalias graves em rotas locais rápidas.
> 
> **Declaração Formal:** Essa árvore de contraste serve **exclusivamente como contra-prova conceitual** e é **descartada** para a entrega final do projeto.

In [ ]:
# ==============================================================================
# 7. ÁRVORE DE CONTRASTE (COM RTT ABSOLUTO)
# ==============================================================================
FEATURES_CONTRASTE = ["rtt_ms", "perda_pct", "timeout_atual", "jitter_ms", "n5_timeout"]

X_train_cont = df_treino[FEATURES_CONTRASTE].copy()
X_val_cont = df_val[FEATURES_CONTRASTE].copy()

clf_contraste = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=20,
    random_state=SEMENTE
)
clf_contraste.fit(X_train_cont, y_train)

regras_contraste = export_text(clf_contraste, feature_names=FEATURES_CONTRASTE)
print("=== ESTRUTURA DA ÁRVORE DE CONTRASTE (COM RTT ABSOLUTO) ===\n")
print(regras_contraste)

print("""
==============================================================================
CONCLUSÃO SOBRE A ÁRVORE DE CONTRASTE:
==============================================================================
• Observe como a árvore de contraste utiliza divisões baseadas em milissegundos absolutos de RTT.
• Esse atalho faz o modelo memorizar que caminhos longos (ex: Europa/EUA com RTT > 150 ms)
  são anomalias, quando na verdade estão perfeitamente saudáveis para a sua distância.
• Essa árvore de contraste É DESCARTADA da entrega final e comprova o acerto
  metodológico do nosso modelo oficial baseado em Z-robusto e aumento percentual!
==============================================================================
""")


## 7. Visualização Gráfica da Árvore Oficial

Geramos o gráfico visual da árvore oficial treinada para documentação e apresentação.

In [ ]:
# ==============================================================================
# 8. PLOTAGEM DA ÁRVORE OFICIAL
# ==============================================================================
plt.figure(figsize=(18, 10), dpi=150)
plot_tree(
    clf_oficial,
    feature_names=FEATURES_OFICIAIS,
    class_names=clf_oficial.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3
)
plt.title("Árvore de Decisão Oficial (Visualização dos Primeiros 3 Níveis)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("arvore_decisao_oficial.png", dpi=150)
plt.show()
print("Gráfico salvo com sucesso em 'arvore_decisao_oficial.png'!")


## 8. Resumo Consolidado para o Diário da Tarefa 3

Execute a célula abaixo para obter o bloco formatado completo para preencher o diário `tarefas/Tarefa3_Arvore_Inicial.md`.

In [ ]:
# ==============================================================================
# 9. RESUMO CONSOLIDADO PARA O DIÁRIO DA TAREFA 3
# ==============================================================================
str_features = ', '.join(FEATURES_OFICIAIS)
matriz_str = df_cm.to_string()

print(f"""
==============================================================================
RESUMO PARA O DIÁRIO DA TAREFA 3 (tarefas/Tarefa3_Arvore_Inicial.md):
==============================================================================

## 1. Colunas que a árvore oficial pode usar:
Colunas usadas: {str_features}

## 2. A árvore:
• Critério: Gini
• Profundidade máxima pedida: 5
• Profundidade obtida: {profundidade_obtida}
• Número de folhas: {num_folhas}
• Semente aleatória: {SEMENTE}

Regras lidas da árvore:
1. SE z_robusto > 3.50 -> FALHA
2. SE perda_pct > 16.67% -> FALHA
3. SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_risco >= 2 E jitter_relativo > 3.00 -> RISCO
4. SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_risco < 2 -> OK

## 3. Leitura do erro (Validação):
• F1 Macro: {f1_macro_val:.4f}
• Matriz 3x3:
{matriz_str}

• Relatório de Classificação:
{report_str}

• Casos divergentes na validação: {len(df_erros)} de {len(df_val)} ({len(df_erros)/len(df_val)*100:.3f}%)

## 4. Árvore de contraste:
• A árvore de contraste utilizou cortes baseados em milissegundos absolutos (rtt_ms),
  evidenciando o atalho indesejado da distância física.
• Declarada formalmente fora da entrega final do preditor.
==============================================================================
""")
